In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR_PATH = '/kaggle/input/competitions/m5-forecasting-accuracy/'

def croston_forecast(ts, extra_periods=28, alpha=0.1):
    """
    Calculates Croston's Method forecast for an intermittent time series.
    ts: 1D numpy array of historical demand (training data)
    extra_periods: How many days into the future to forecast
    alpha: The smoothing parameter for both demand size and inter-arrival time
    """
    d = np.array(ts)
    cols = len(d)
    
    # Initialize arrays
    a = np.zeros(cols) # Smoothed demand size
    p = np.zeros(cols) # Smoothed inter-arrival time
    f = np.zeros(cols) # Forecast
    
    # Find the first period with actual demand to initialize
    non_zero_idx = np.where(d > 0)[0]
    if len(non_zero_idx) == 0:
        return np.zeros(extra_periods) # If it never sold, predict 0
    
    first_idx = non_zero_idx[0]
    
    a[first_idx] = d[first_idx]
    p[first_idx] = 1.0 
    f[first_idx] = a[first_idx] / p[first_idx]
    
    q = 1 # Time since last demand
    
    # Run the Croston's loop
    for i in range(first_idx + 1, cols):
        if d[i] > 0:
            a[i] = alpha * d[i] + (1 - alpha) * a[i-1]
            p[i] = alpha * q + (1 - alpha) * p[i-1]
            f[i] = a[i] / p[i]
            q = 1
        else:
            a[i] = a[i-1]
            p[i] = p[i-1]
            f[i] = f[i-1]
            q += 1
            
    # The final forecast is the last calculated ratio, projected forward
    return np.full(extra_periods, f[-1])

def run_crostons_experiment():
    print("--- Loading Data ---")
    sales = pd.read_csv(INPUT_DIR_PATH + 'sales_train_validation.csv')
    
    # Define the exact same time splits as LightGBM
    train_cols = [f'd_{i}' for i in range(1, 1886)] # Days 1 to 1885
    val_cols = [f'd_{i}' for i in range(1886, 1914)] # Days 1886 to 1913
    
    results = []
    
    print(f"--- Running Croston's Method on {len(sales)} products ---")
    # Loop through every single product in the dataset
    for index, row in sales.iterrows():
        item_id = row['id']
        
        # 1. Get training history and actual validation targets
        train_history = row[train_cols].values.astype(np.float32)
        actual_val_demand = row[val_cols].values.astype(np.float32)
        
        # 2. Generate the 28-day forecast using Croston's math
        # alpha=0.1 is standard for intermittent retail demand
        predicted_val_demand = croston_forecast(train_history, extra_periods=28, alpha=0.1)
        
        # 3. Calculate Item-Level RMSE
        squared_errors = (predicted_val_demand - actual_val_demand) ** 2
        rmse = np.sqrt(np.mean(squared_errors))
        
        results.append({
            'id': item_id,
            'croston_rmse': rmse
        })
        
        if (index + 1) % 5000 == 0:
            print(f"Processed {index + 1} / {len(sales)} items...")

    # Save the results to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv('croston_item_level_rmse.csv', index=False)
    
    global_rmse = np.sqrt(np.mean(results_df['croston_rmse']**2))
    print(f"\nSuccessfully saved item-level RMSE to 'croston_item_level_rmse.csv'")
    print(f"Our final GLOBAL Croston's validation RMSE is {global_rmse:.3f}")

# Execute the experiment
run_crostons_experiment()

--- Loading Data ---
--- Running Croston's Method on 30490 products ---
Processed 5000 / 30490 items...
Processed 10000 / 30490 items...
Processed 15000 / 30490 items...
Processed 20000 / 30490 items...
Processed 25000 / 30490 items...
Processed 30000 / 30490 items...

Successfully saved item-level RMSE to 'croston_item_level_rmse.csv'
Our final GLOBAL Croston's validation RMSE is 2.299
